# Drop weak components: the object filter (CPU)

Import this notebook into Kaggle and press **Save & Run All**. None of the cells asks for input. With the default `MODEL = "r34"` no cell needs editing. To use another model, change only `MODEL` in the settings cell and attach its cache:

| `MODEL` | Encoder | Save the training notebook as | Checkpoint |
| --- | --- | --- | --- |
| `r34` (default) | `resnet34` | `train-unet-r34-crops-80ep` | `unet_r34_crops.pt` |
| `r50` | `resnet50` | `train-unet-r50-crops` | `unet_r50_crops.pt` |
| `effb3` | `efficientnet-b3` | `train-unet-effb3-crops` | `unet_effb3_crops.pt` |
| `r34_v2` | `resnet34` | `train-unet-r34-v2` | `unet_r34_v2.pt` |
| `r34_v2_agree` | `resnet34` | `train-unet-r34-v2-agree` | `unet_r34_v2_agree.pt` |

Today every connected component above `min_area` is emitted, however unsure the network was. This notebook is step B1 of `docs/ROADMAP_V2.md`: it drops components with a low probability, from the maps of `notebooks/export_probs.ipynb`, without a GPU. It moves one axis at a time, never a fine two-axis grid:

1. **Object filter:** the mean, maximum or 90th percentile probability of a component, at the current post-processing setting.
2. **`min_area`:** lower values, since the filter may now do part of the area cut's job, at the object filter from step 1.
3. **Pixel threshold:** above 0.60 too, without the filter.

Picking the best row of those tables would overstate the gain. So the verdict is a cross-fit over all of them on 5 partitions of the dates: each partition chooses on some dates and scores on the others, and those scores go through the paired bootstrap against the current setting. The rule is the roadmap's: a gain of at least 0.005 in at least 90% of the resamples, in at least 4 of the 5 partitions. Step 2 starts from the cut chosen on all files in step 1, so even this leans slightly high.

It reads only the holdout side of the cache. It does not train, does not read the test images, does not write `submission.csv` and does not submit to Kaggle.

**Attach exactly these Kaggle inputs**

1. The competition `filament-segmentation-2026` (train images and train JSON).
2. The dataset `ignacioaguirrecandia/solar-code` (the repo zip), a version that contains `filament.components`. An older version stops at the imports.
3. The private dataset `solar-probs-<MODEL>`, made from the output of `export_probs.ipynb` for this model.

**Settings**

- Accelerator: **None**. This notebook does not use a GPU.
- Internet: **Off** is enough.

**Output:** `/kaggle/working/holdout_object_filter_<MODEL>.csv`, small and with no image or mask data, so it can go in `results/`. The running time is not measured yet [VERIFY]; a progress line is printed every 20 frames of each step.

## 1. `src` on the path

`src` is the directory that contains `filament/crop_training.py`. A checkout in the working directory is used as it is. Otherwise the walk goes through `/kaggle/input` at any depth, skips every `competitions` directory, and takes the shallowest match. It is used in place and not copied.

In [ ]:
import os
import sys
from pathlib import Path


def _local_src() -> Path | None:
    """A checkout that already has the crop modules (a run off Kaggle)."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        src = candidate / "src"
        if (src / "filament" / "crop_training.py").is_file():
            return src
    return None


def _private_dataset_src(root: Path | None = None) -> Path | None:
    """Return the shallowest private-dataset ``src`` under ``root``.

    ``filament/crop_training.py`` may sit at any depth. Directories named
    ``competitions`` are not walked, so the MAGFiLO tree is left alone.
    """
    root = Path("/kaggle/input") if root is None else root
    if not root.is_dir():
        return None
    marker = Path("filament") / "crop_training.py"
    matches: list[Path] = []
    for dirpath, dirnames, _filenames in os.walk(root, topdown=True):
        dirnames[:] = [name for name in dirnames if name != "competitions"]
        current = Path(dirpath)
        if current.name != "src" or not (current / marker).is_file():
            continue
        matches.append(current)
        dirnames.clear()
    if not matches:
        return None
    return min(
        matches,
        key=lambda path: (len(path.relative_to(root).parts), path.as_posix()),
    )


def _put_on_path(src: Path) -> None:
    text = str(src)
    if text not in sys.path:
        sys.path.insert(0, text)
    current = os.environ.get("PYTHONPATH", "")
    parts = [part for part in current.split(os.pathsep) if part]
    if text not in parts:
        os.environ["PYTHONPATH"] = os.pathsep.join([text, *parts]) if parts else text


def _prepare_src() -> Path:
    """Put the repo's ``src`` on ``sys.path`` and return it.

    A checkout in the working directory is used as it is. Otherwise the
    private dataset's ``src`` is used in place. It is not copied to
    ``/kaggle/working``: a copy there would be saved with this notebook's
    output and found again by a later notebook that attaches that output.
    """
    local = _local_src()
    if local is not None:
        _put_on_path(local)
        return local
    source = _private_dataset_src()
    if source is None:
        raise FileNotFoundError(
            "Could not find src/filament/crop_training.py in the working "
            "directory or under /kaggle/input (any depth outside "
            "competitions). Attach the dataset ignacioaguirrecandia/solar-code, "
            "for example "
            "/kaggle/input/datasets/ignacioaguirrecandia/solar-code/"
            "Solar-Filament-Segmentation-main/src. If it is attached, its "
            "version predates the crop modules: upload a new version of the "
            "dataset from the branch that has them. Do not make the GitHub "
            "repository public."
        )
    _put_on_path(source)
    return source


SRC = _prepare_src()
print(f"src={SRC}")
PYTHON = sys.executable

## 2. Settings and the cache

- `BASE` is the current setting (`docs/PROJECT_REPORT.md`, row 12). Every difference is measured against it.
- The grids come from `filament.components`. They are not tuned.
- The cache has to come from exactly this checkpoint: its manifest goes through the same guard as a weights file. Leave `ALLOW_UNVERIFIED_WEIGHTS` as `False`.

In [ ]:
from filament.annotations import (
    TRAIN_ANNOTATIONS_PATH,
    TRAIN_IMAGES_DIR,
    load_train_annotations,
)
from filament.axis_report import (
    axis_rows,
    crossfit_rows,
    format_axis_rows,
    write_axis_rows,
)
from filament.checkpoint import crop_checkpoint, verify_and_report
from filament.components import (
    HIGH_THRESHOLDS,
    LOW_MIN_AREAS,
    NO_OBJECT_FILTER,
    OBJECT_CUTS,
    OBJECT_STATISTICS,
    ObjectCut,
    per_file_object_counts,
)
from filament.crop_data import read_gray
from filament.gap_analysis import CURRENT_POINT, holdout_dates
from filament.probcache import (
    ProbCache,
    find_cache_root,
    manifest_fingerprint,
    read_manifest,
    split_dir,
)
from filament.stats import plateau_center, weighted_pq
from filament.sweep import GridPoint, count_views, per_file_counts
from filament.train_data import split_training_images

# ---- Which model: a name of filament.checkpoint.CROP_CHECKPOINTS --------------
MODEL = "r34"
EXPECTED = crop_checkpoint(MODEL)

# ---- The sweep ---------------------------------------------------------------
BASE = CURRENT_POINT
ALLOW_UNVERIFIED_WEIGHTS = False

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_PATH = Path(f"/kaggle/working/holdout_object_filter_{MODEL}.csv")

CACHE_ROOT = find_cache_root(MODEL, INPUT_ROOT)
print(f"cache={CACHE_ROOT}")
verify_and_report(
    manifest_fingerprint(read_manifest(CACHE_ROOT, MODEL)),
    allow_unverified=ALLOW_UNVERIFIED_WEIGHTS,
    expected=EXPECTED,
)

## 3. Date holdout

The same split as the training notebook: the most recent 20% of calendar dates (142 files with 231 annotator views on the current data). Every holdout file has to be in the cache.

In [ ]:
IMAGE_DIR = Path(TRAIN_IMAGES_DIR)
ANNOTATIONS_PATH = Path(TRAIN_ANNOTATIONS_PATH)
if not ANNOTATIONS_PATH.is_file() or not IMAGE_DIR.is_dir():
    raise FileNotFoundError(
        "Train data was not found. Checked "
        f"images={IMAGE_DIR} and annotations={ANNOTATIONS_PATH}. "
        "Attach the competition filament-segmentation-2026 to this notebook."
    )
loaded = load_train_annotations(ANNOTATIONS_PATH)
holdout = split_training_images(loaded.images).holdout
holdout_views = count_views(holdout)
print(f"holdout files={len(holdout)} annotator views={holdout_views}")
if (len(holdout), holdout_views) != (142, 231):
    print(
        "WARNING: the holdout is not 142 files with 231 views, the numbers the "
        "training notebook used. [VERIFY] the data version."
    )
cache = ProbCache(split_dir(CACHE_ROOT, MODEL, "holdout"))
missing = [image.file_name for image in holdout if image.file_name not in cache]
if missing:
    raise RuntimeError(
        f"{len(missing)} holdout files are not in the cache, for example {missing[0]}."
    )

## 4. One axis at a time

The scoring is `filament.sweep`: pooled per-view PQ, kept per file. The maps are the flip-TTA means from the cache. Each table shows the paired difference from `BASE` over the same resampled files.

Step 2 uses the center of the plateau of the mean-probability filter from step 1, not its peak: `filament.stats.plateau_center`.

In [ ]:
def _read(image):
    return read_gray(IMAGE_DIR / image.file_name, (image.height, image.width))


def _load(image):
    return cache.mean(image.file_name)


def _progress(index, total, image):
    if index == 1 or index % 20 == 0 or index == total:
        print(f"  {index}/{total} {image.file_name}", flush=True)


# Step 1: the object filter at the current setting.
print("step 1: object filter")
cuts = [NO_OBJECT_FILTER] + [
    ObjectCut(statistic, cut) for statistic in OBJECT_STATISTICS for cut in OBJECT_CUTS
]
step1 = per_file_object_counts(holdout, _read, _load, BASE, cuts, progress=_progress)
base_counts = step1.counts[NO_OBJECT_FILTER]
print(f"pooled PQ at {BASE}: {weighted_pq(base_counts):.4f}")

# Step 2: min_area, at the center of the plateau of the mean-probability filter.
mean_cuts = [ObjectCut("mean", cut) for cut in OBJECT_CUTS]
center = plateau_center([weighted_pq(step1.counts[cut]) for cut in mean_cuts])
chosen_cut = mean_cuts[center]
print(f"step 2: min_area at {chosen_cut}")
step2 = {}
for area in LOW_MIN_AREAS:
    point = GridPoint(BASE.threshold, BASE.close_size, area)
    result = per_file_object_counts(holdout, _read, _load, point, [chosen_cut])
    step2[f"{chosen_cut}, min_area {area}"] = result.counts[chosen_cut]

# Step 3: the pixel threshold, without the filter.
print("step 3: pixel threshold")
points = [
    GridPoint(threshold, BASE.close_size, BASE.min_area)
    for threshold in HIGH_THRESHOLDS
]
step3 = per_file_counts(holdout, _read, _load, points, progress=_progress)

rows = []
rows += axis_rows("object", base_counts, step1.counts)
rows += axis_rows("min_area", base_counts, step2)
rows += axis_rows(
    "threshold",
    base_counts,
    {f"threshold {p.threshold:g}": step3.counts[p] for p in points},
)
print(format_axis_rows(rows))

## 5. The verdict

A cross-fit over every setting above, including the current one, so a partition may keep it. Read the last line: `ADOPT` or `DO NOT ADOPT`. If it is `DO NOT ADOPT`, keep the current setting, however good a single row looks above.

Add one row to `docs/RESULTS.md` from this output and put the CSV in `results/`.

In [ ]:
dates = holdout_dates(holdout)
every_setting = {"current": base_counts}
every_setting.update({str(cut): step1.counts[cut] for cut in cuts[1:]})
every_setting.update(step2)
every_setting.update({f"threshold {p.threshold:g}": step3.counts[p] for p in points})
verdict_rows, adopted = crossfit_rows("all", base_counts, every_setting, dates)
print(format_axis_rows(verdict_rows))
write_axis_rows(OUTPUT_PATH, rows + verdict_rows)
print(f"wrote {OUTPUT_PATH}")
print("This notebook read no test image, wrote no submission and did not submit.")